In [ ]:
import torch
import pandas as pd
import numpy as np
import plotly.express as px
from transformer_time_series_encoder_only import (
    StockInformerEncoderOnly, create_dataloaders, TrainConfig,
    train_model, inverse_transform, build_target,init_weights
)
import random

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
# ==============================
# Load data and define config
# ==============================

csv_path = r"D:\Quan\Quants\Neural Network\financial_attention\1h_data_20220101_20250601.csv"
closes = pd.read_csv(csv_path, index_col=0, parse_dates=True)[['SOL', 'ETH', 'BTC','ADA','XRP','LTC','TRX','LINK','DOT','DOGE']]

config = {
    "d_input": len(closes.columns),
    "d_model": 64,
    "n_heads": 4,
    "d_ff": 256,
    "enc_layers": 3,
    "dropout": 0.05,
    "distill": False,
    "enc_len": 96,
    "pred_len": 1,  # Always 1-step ahead
    "factor": 5,
    "use_time_embedding": False,
    "norm_mode": "post",
    "attention_type": "prob",   # Options: "prob" or "full"
    "target_type": "price",    # Options: "change" or "price"
}
set_seed(42)
train_loader, val_loader, scaler, asset_idx = create_dataloaders(
    closes, enc_len=config["enc_len"], pred_len=config["pred_len"],
    batch_size=32, val_ratio=0.1, asset_name="SOL"
)
print(f"✅ Data ready: {len(train_loader.dataset)} training samples, {len(val_loader.dataset)} validation samples.")

✅ Data ready: 26275 training samples, 2931 validation samples.


In [1]:
# ==============================
# Initialize and train model
# ==============================
model = StockInformerEncoderOnly(config, asset_index=asset_idx)

model.apply(init_weights)

tcfg = TrainConfig(learning_rate=1e-4, weight_decay=0.01, max_steps=10000, warmup_steps=200, use_amp=True, device="cuda",
                   patience = 15, min_delta=0.0001)
model, train_hist, val_hist, steps_hist, best_val_loss, best_val_step = train_model(
    model, train_loader, val_loader, tcfg, asset_index=asset_idx
)

print(f"✅ Training done. Best validation loss: {best_val_loss:.6f} at step {best_val_step}")

Traceback (most recent call last):
  File "_pydevd_bundle\\pydevd_cython.pyx", line 1609, in _pydevd_bundle.pydevd_cython.handle_exception
  File "C:\Users\QuanNguyen.DESKTOP-JSKT55M\AppData\Roaming\Python\Python312\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 2188, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, trace_suspend_type, from_this_thread, frames_tracker)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\QuanNguyen.DESKTOP-JSKT55M\AppData\Roaming\Python\Python312\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 2257, in _do_wait_suspend
    notify_event.wait(wait_timeout)
  File "c:\Users\QuanNguyen.DESKTOP-JSKT55M\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 655, in wait
    signaled = self._cond.wait(timeout)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\QuanNguyen.DESKTOP-JSKT55M\AppData\Local\P

NameError: name 'StockInformerEncoderOnly' is not defined

In [4]:
# ==============================
# Evaluate and plot results
# ==============================
device = torch.device(tcfg.device)
model = model.to(device)
preds, targets = [], []

with torch.no_grad():
    for val_seqs, val_times in val_loader:
        val_seqs, val_times = val_seqs.to(device), val_times.to(device)
        y_pred = model(val_seqs, val_times)
        y_true = build_target(val_seqs, asset_idx, model.target_type)
        preds.append(y_pred.cpu().numpy())
        targets.append(y_true.cpu().numpy())

preds = np.concatenate(preds).flatten()
targets = np.concatenate(targets).flatten()

if config["target_type"] == "price":
    preds_plot = inverse_transform(preds, scaler, asset_idx, config["d_input"])
    targets_plot = inverse_transform(targets, scaler, asset_idx, config["d_input"])
    y_label, title = "Price", "Predicted vs Actual Price (Validation Set)"
else:
    preds_plot, targets_plot = preds, targets
    y_label, title = "Log Return (1h)", "Predicted vs Actual Log-Return (Validation Set)"

df_plot = pd.DataFrame({
    "Time Step": np.arange(len(targets_plot)),
    "Actual": targets_plot,
    "Predicted": preds_plot
})
df_plot = df_plot.melt(id_vars="Time Step", value_vars=["Actual", "Predicted"], var_name="Type", value_name=y_label)

fig = px.line(df_plot, x="Time Step", y=y_label, color="Type", title=title)
fig.show()